[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/automatizacion/01-n8n-ia.ipynb)

# Automatización con n8n e IA

> Aprende a integrar Claude con n8n para crear flujos de automatización
> sin código que procesan datos, clasifican contenido y generan respuestas automáticas.

## Objetivos
- Entender la arquitectura de n8n y sus nodos de IA
- Simular workflows de n8n implementándolos en Python
- Construir un pipeline de clasificación y respuesta automática de emails
- Implementar webhook handlers para integrar Claude con eventos externos

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic flask pandas --quiet

## 2. Arquitectura de n8n con IA

In [ ]:
import anthropic
import json
import pandas as pd
from datetime import datetime
from typing import Optional

client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

print("""=== n8n + IA: ARQUITECTURA DE WORKFLOWS ===

n8n es una plataforma de automatización visual de código abierto.
Permite conectar servicios y añadir inteligencia artificial sin programar.

Nodos relevantes para IA:
  📥 Trigger nodes: Webhook, Email, Cron, Google Sheets
  🤖 AI nodes: OpenAI, Anthropic, HuggingFace
  🔀 Logic nodes: IF/Switch, Merge, Split
  📤 Output nodes: Gmail, Slack, Airtable, HTTP Request

Workflows típicos con Claude:
  1. Email → Clasificar → Responder/Escalar → CRM
  2. Formulario → Extraer datos → Validar → BD
  3. RSS Feed → Resumir → Newsletter → Enviar
  4. Slack mensaje → Analizar → Responder → Ticketing

Este notebook simula esos workflows en Python puro.
En producción: instala n8n con `npx n8n` o `docker run n8nio/n8n`
""")

## 3. Workflow 1: Clasificación y respuesta automática de emails

In [ ]:
# Simular nodo Trigger: Email recibido
EMAILS_ENTRANTES = [
    {"id": "e001", "de": "cliente@empresa.com", "asunto": "Problema con mi factura de noviembre",
     "cuerpo": "Buenos días, he recibido una factura incorrecta por importe de 450€ cuando lo acordado era 380€. Por favor revisen."},
    {"id": "e002", "de": "ventas@proveedor.es", "asunto": "Propuesta de colaboración comercial",
     "cuerpo": "Estimados, somos una empresa de servicios de marketing digital y nos gustaría presentarles nuestra propuesta."},
    {"id": "e003", "de": "maria.garcia@cliente.com", "asunto": "¿Cómo exporto mis datos?",
     "cuerpo": "Hola, llevo una hora intentando exportar mis datos en formato CSV y no encuentro la opción. ¿Podéis ayudarme?"},
    {"id": "e004", "de": "critico@usuario.es", "asunto": "Voy a poner una denuncia",
     "cuerpo": "Esto es un FRAUDE. Lleváis 3 semanas sin resolver mi problema y encima me habéis cobrado dos veces. Exijo una solución INMEDIATA."},
]

def nodo_clasificar_email(email: dict) -> dict:
    """Nodo IA: clasifica el email y determina prioridad y acción."""
    prompt = f"""Clasifica este email de soporte al cliente.

Asunto: {email['asunto']}
Cuerpo: {email['cuerpo']}

Responde SOLO con JSON:
{{"categoria": "facturacion|soporte_tecnico|comercial|queja_urgente|otro",
  "prioridad": "alta|media|baja",
  "sentimiento": "positivo|neutro|negativo|muy_negativo",
  "requiere_humano": true/false,
  "departamento": "facturacion|soporte|comercial|direccion"}}"""

    resp = client.messages.create(
        model=MODELO, max_tokens=150,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text.strip()
    if "```" in resp:
        resp = resp.split("```")[1].lstrip("json")
    clasificacion = json.loads(resp)
    return {**email, **clasificacion}

def nodo_generar_respuesta(email_clasificado: dict) -> dict:
    """Nodo IA: genera respuesta automática si no requiere humano."""
    if email_clasificado.get("requiere_humano"):
        return {**email_clasificado, "respuesta_auto": None,
                "accion": f"ESCALAR a {email_clasificado['departamento']}"}

    prompt = f"""Genera una respuesta profesional y empática para este email.

Asunto: {email_clasificado['asunto']}
Mensaje: {email_clasificado['cuerpo']}
Categoría: {email_clasificado['categoria']}

La respuesta debe:
- Ser en español, profesional y cordial
- Confirmar que hemos recibido su consulta
- Dar una respuesta útil si es posible
- Máximo 3 párrafos cortos"""

    respuesta = client.messages.create(
        model=MODELO, max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text
    return {**email_clasificado, "respuesta_auto": respuesta, "accion": "RESPONDER_AUTO"}

print("=== EJECUTANDO WORKFLOW DE EMAILS ===")
resultados = []
for email in EMAILS_ENTRANTES:
    clasificado = nodo_clasificar_email(email)
    procesado = nodo_generar_respuesta(clasificado)
    resultados.append(procesado)
    print(f"\n[{procesado['id']}] {procesado['asunto'][:45]}")
    print(f"  → {procesado['categoria']} | prioridad:{procesado['prioridad']} | {procesado['accion']}")
    if procesado.get('respuesta_auto'):
        print(f"  Respuesta: {procesado['respuesta_auto'][:100]}...")

## 4. Workflow 2: Procesamiento de formularios web

In [ ]:
# Simular webhook de formulario web
FORMULARIOS = [
    {"nombre": "Carlos Ruiz", "empresa": "Agencia Digital SL", "empleados": "15",
     "mensaje": "Necesito automatizar el reporte semanal de redes sociales y envío de newsletter"},
    {"nombre": "Ana Martínez", "empresa": "Clínica Salud Plus", "empleados": "50",
     "mensaje": "Queremos digitalizar la gestión de citas y recordatorios automáticos por SMS"},
]

def procesar_formulario(formulario: dict) -> dict:
    """Simula el workflow completo de procesamiento de un lead."""
    prompt = f"""Analiza este lead de formulario de contacto B2B.

Empresa: {formulario['empresa']} ({formulario['empleados']} empleados)
Necesidad: {formulario['mensaje']}

Responde con JSON:
{{"sector": "<sector de la empresa>",
  "caso_uso": "<descripción concisa del caso de uso>",
  "potencial": "alto|medio|bajo",
  "siguiente_paso": "<acción comercial recomendada>",
  "email_bienvenida": "<email corto de bienvenida personalizado>"}}"""

    resp = client.messages.create(
        model=MODELO, max_tokens=400,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text.strip()
    if "```" in resp:
        resp = resp.split("```")[1].lstrip("json")
    analisis = json.loads(resp)
    return {**formulario, **analisis, "fecha": datetime.now().isoformat()}

print("=== PROCESAMIENTO DE FORMULARIOS ===")
for f in FORMULARIOS:
    resultado = procesar_formulario(f)
    print(f"\n{resultado['nombre']} — {resultado['empresa']}")
    print(f"  Sector: {resultado['sector']} | Potencial: {resultado['potencial']}")
    print(f"  Caso uso: {resultado['caso_uso']}")
    print(f"  Siguiente paso: {resultado['siguiente_paso']}")
    print(f"  Email: {resultado['email_bienvenida'][:100]}...")

## 5. Webhook handler para integración en tiempo real

In [ ]:
# Código del servidor webhook — guárdalo en webhook_ia.py
WEBHOOK_CODE = '''
from flask import Flask, request, jsonify
import anthropic
import json

app = Flask(__name__)
client = anthropic.Anthropic()

@app.route("/webhook/email", methods=["POST"])
def procesar_email():
    """Webhook que n8n llama cuando llega un email."""
    data = request.json
    asunto = data.get("subject", "")
    cuerpo = data.get("body", "")

    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=200,
        messages=[{"role": "user", "content":
            f"Clasifica este email. Asunto: {asunto}. Cuerpo: {cuerpo[:300]}"
            " Responde JSON: {categoria, prioridad, requiere_humano}"}]
    ).content[0].text

    return jsonify({"clasificacion": json.loads(resp), "email_id": data.get("id")})

@app.route("/webhook/formulario", methods=["POST"])
def procesar_formulario():
    """Webhook para formularios de contacto."""
    data = request.json
    prompt = f"Analiza este lead: {json.dumps(data)}. Devuelve JSON con: sector, potencial, siguiente_paso."
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001", max_tokens=200,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text
    return jsonify(json.loads(resp))

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5678)
'''

with open("webhook_ia.py", "w", encoding="utf-8") as f:
    f.write(WEBHOOK_CODE)

print("✓ webhook_ia.py generado")
print("Para arrancar: python webhook_ia.py")
print("Para exponer externamente: ngrok http 5678")
print("""
=== CONFIGURAR EN n8n ===

1. Instala n8n: npx n8n
2. Crea workflow con nodo 'Webhook'
3. URL del webhook: http://localhost:5678/webhook/email
4. Conecta al nodo 'HTTP Request' apuntando a tu servidor Python
5. Añade nodos de salida: Gmail (responder), Slack (notificar), Airtable (guardar)

Flujo en n8n:
  [Gmail Trigger] → [HTTP Request → webhook_ia.py] → [IF prioridad=alta] → [Slack]
                                                    → [ELSE] → [Gmail responder auto]
""")